In [2]:
import cv2
import numpy as np

def ideal_lowpass_filter(image, d0):
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2 

    dft = np.fft.fft2(image.astype(np.float32))
    dft_shift = np.fft.fftshift(dft)

    mask = np.zeros((rows, cols), np.uint8)
    for u in range(rows):
        for v in range(cols):
            dist = np.sqrt((u - crow)**2 + (v - ccol)**2)
            if dist <= d0:
                mask[u, v] = 1  # Пропускаем частоты ниже порога D0

    fshift = dft_shift * mask

    # Обратное преобразование Фурье
    f_ishift = np.fft.ifftshift(fshift) # Децентрирование
    img_back = np.fft.ifft2(f_ishift)

    img_back = np.abs(img_back)

    return img_back.astype(np.uint8)

def on_trackbar(val):
    d0 = cv2.getTrackbarPos('D0 (Cutoff)', 'Ideal Lowpass Filter')
    if d0 < 1: d0 = 1
    
    filtered_img = ideal_lowpass_filter(original_img, d0)
    
    dft = np.fft.fftshift(np.fft.fft2(original_img))
    magnitude_spectrum = 20 * np.log(np.abs(dft) + 1)
    
    combined = np.hstack((original_img, filtered_img))
    cv2.imshow('Ideal Lowpass Filter', combined)

filename = '1.jpg' 
original_img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

if original_img is None:
    print(f"Ошибка загрузки файла {filename}")
else:
    cv2.namedWindow('Ideal Lowpass Filter')
    max_d0 = min(original_img.shape) // 2
    cv2.createTrackbar('D0 (Cutoff)', 'Ideal Lowpass Filter', 30, max_d0, on_trackbar)
    
    on_trackbar(30)
    while True:
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cv2.destroyAllWindows()